###Build Constructors Standings

**Sources**\
    1. fact_session_results\
    2. dim_constructors

**Output Columns**\
    1. season\
    2. constructor_id\
    4. nationality\
    5. race starts\
    6. total points\
    7. no of wins\
    8. no of podiums\
    9. standing position

In [0]:
%sql
select * from formula1.gold.fact_session_results limit 5

In [0]:
%sql
ALTER TABLE formula1.gold.fact_session_results SET TBLPROPERTIES ('delta.columnMapping.mode' = 'name'); ALTER TABLE formula1.gold.fact_session_results RENAME COLUMN is_podimun TO is_podium;

In [0]:
%sql
select * from formula1.gold.dim_constructors limit 5

In [0]:
%sql
SELECT *
    FROM formula1.gold.vw_driver_standings

In [0]:
%sql
CREATE OR REPLACE VIEW formula1.gold.vw_constructor_standings
AS
WITH constructor_session_summary
AS
    (
        SELECT r.season,
            c.constructor_id,
            c.name as constructor_name,
            c.nationality,
            COUNT(*) AS race_starts,
            SUM(r.points) AS total_points,
            count_if(r.is_win) AS wins,
            count_if(r.is_podium) AS podiums
        FROM formula1.gold.fact_session_results r
        JOIN formula1.gold.dim_constructors c
            ON r.constructor_id = c.constructor_id
        GROUP BY r.season,
            c.constructor_id,
            c.name,
            c.nationality  
            ) 
    SELECT  season,
            constructor_id,
            constructor_name,
            nationality,
            RANK() OVER(PARTITION BY season ORDER BY total_points DESC, wins DESC) AS standings,
            race_starts,
            total_points,
            wins,
            podiums
        FROM constructor_session_summary

In [0]:
%sql
SELECT * FROm formula1.gold.vw_constructor_standings